# Standalone xNovo-DDA prediction and two-level CCT quality control

This notebook accompanies **False discovery rate control enables trustworthy
deep-learning-based de novo peptide sequencing** (Liang et al.; Figs. 1 and 3,
Supplementary Table 2 and Supplementary Note 2).

Start with the README in this directory for data and pretrained-weight downloads,
installation and input formats. Launch JupyterLab from this directory, edit
`config.yaml`, then run the cells in order. All implementation code is included
in `cct_qc`; no xNovo training package or NovoQC installation is required.

The xNovo-DDA sequencing checkpoint remains fixed. Separate strict and relaxed
QC scoring models are trained using reference-labeled PSMs. Configure the PSM
and peptide target FDR thresholds and a new output directory in `config.yaml`.
The analysis ends with `psms.csv` and `peptides.csv`; protein inference and
quantification are not required. This example runs the method; reproducing a
specific paper result also requires that experiment's data, checkpoint and settings.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

from cct_qc import features, qc
from cct_qc.data import mgf_batches

with open("config.yaml") as handle:
    config = yaml.safe_load(handle)

device = config["device"]
output = Path(config["output"])
output.mkdir(parents=True, exist_ok=False)
model_dir = output / "models"
q_psm = float(config["psm_target_fdr"])
q_peptide = float(config["peptide_target_fdr"])
if not 0 < q_psm < 1 or not 0 < q_peptide < 1:
    raise ValueError("Target FDR thresholds must be between zero and one")
seed, epochs = int(config["seed"]), int(config["epochs"])
columns = features.FEATURE_COLUMNS


## 1. Generate or read paired features

For each predicted peptide and spectrum, the target and counterpart use the
same inference cross-attention matrix. For peptide length `L`, the two row slices
are `0:L` and `1:1+L`. Counterpart generation does not use training-mode dropout.

For the annotated GraphNovo benchmark, MGF `SEQ` supplies a reference peptide
used to assess the de novo prediction. It is not supplied to the sequencing
model. Set `use_mgf_reference: false` for unannotated MGF inputs. Missing
reference labels remain missing; they are not assigned to the incorrect class.


In [ ]:
if config["generate_features"]:
    sequencing_model, model_config = features.load_model(
        config["checkpoint"], config["model_config"], device)
    batches = mgf_batches(
        config["samples"], model_config, config["batch_size"],
        config["use_mgf_reference"], config.get("reference_csv"))
    frame = pd.DataFrame(features.extract_features(
        sequencing_model, batches, model_config["residues"], device))
    feature_path = output / "paired_features.csv"
    frame.to_csv(feature_path, index=False)
    del sequencing_model
else:
    feature_path = Path(config["feature_csv"])

frame = features.read_features(feature_path)
reference = frame.loc[frame["correct"].notna()].copy()
if reference.empty:
    raise ValueError("Reference-labeled target/counterpart pairs are required for fitting and calibration")
reference = qc.prepare(reference)
candidates = qc.prepare(frame)
display(pd.DataFrame({
    "PSMs": [len(frame) // 2, len(reference) // 2],
}, index=["All candidates", "Reference-labeled candidates"]))


## 2. PSM scoring, FDR adjustment and filtering

Train the strict and relaxed PSM scoring models on the reference-labeled pairs.
Their equally weighted scores define the ranking. Estimate the PSM FDR adjustment
factor `gamma_psm` from the reference set, then apply the fitted scoring models
and adjusted count-based FDR estimate to all candidates.


In [ ]:
qc.fit_stage(reference, model_dir / "psm", columns, seed, epochs, device)
reference_psm = qc.score(reference, model_dir / "psm", columns, device)
gamma_psm = qc.calibrate(reference_psm, q_psm)
reference_curve, reference_selected = qc.select(reference_psm, gamma_psm, q_psm)

all_psm = qc.score(candidates, model_dir / "psm", columns, device)
psm_curve, selected_psm_pairs = qc.select(all_psm, gamma_psm, q_psm)
accepted_psms = selected_psm_pairs.loc[~selected_psm_pairs["is_decoy"]].copy()
psm_curve.to_csv(output / "psm_scores.csv", index=False)
accepted_psms.to_csv(output / "psms.csv", index=False)
print(f"PSM target FDR threshold: {q_psm:.1%}; gamma: {gamma_psm:.6g}")
print(f"Accepted target PSMs: {len(accepted_psms):,}")


## 3. Construct peptide representatives and fit peptide scoring models

Recover both rows of every PSM pair selected at the PSM stage. Retain the highest
PSM-score representative for each peptide key, then train the peptide-level
strict and relaxed scoring models on the reference-labeled representatives.


In [ ]:
reference_peptides = qc.peptide_input(reference_curve, reference_selected)
all_peptides = qc.peptide_input(psm_curve, selected_psm_pairs)
if reference_peptides.empty:
    raise ValueError("No reference PSMs passed the PSM stage; peptide models cannot be fitted")
qc.fit_stage(reference_peptides, model_dir / "peptide", columns, seed, epochs, device)
reference_peptides = qc.score(reference_peptides, model_dir / "peptide", columns, device)
gamma_peptide = qc.calibrate(reference_peptides, q_peptide)


## 4. Peptide-level FDR control and result export

Apply the peptide scoring models and adjustment factor to the peptide
representatives. The exported PSMs passed PSM-level QC; the exported peptides
passed both stages. These peptide results can be used independently of protein
inference or quantification, including as inputs to SAP analyses.


In [ ]:
if len(all_peptides):
    all_peptides = qc.score(all_peptides, model_dir / "peptide", columns, device)
peptide_curve, selected_peptide_pairs = qc.select(all_peptides, gamma_peptide, q_peptide)
accepted_peptides = selected_peptide_pairs.loc[~selected_peptide_pairs["is_decoy"]].copy()
peptide_curve.to_csv(output / "peptide_scores.csv", index=False)
accepted_peptides.to_csv(output / "peptides.csv", index=False)

fitted = dict(feature_columns=columns, q_psm=q_psm, q_peptide=q_peptide,
              gamma_psm=gamma_psm, gamma_peptide=gamma_peptide,
              seed=seed, acquisition="dda")
(model_dir / "config.json").write_text(json.dumps(fitted, indent=2) + "\n")

def summarize(accepted, curve, threshold, gamma):
    observed = (1 - accepted["correct"].astype(float).mean()
                if len(accepted) and accepted["correct"].notna().all() else np.nan)
    return dict(target_fdr_threshold=threshold, gamma=gamma, accepted=len(accepted),
                observed_fdp=observed)

summary = pd.DataFrame([
    summarize(accepted_psms, psm_curve, q_psm, gamma_psm),
    summarize(accepted_peptides, peptide_curve, q_peptide, gamma_peptide),
], index=["PSM", "Peptide"])
summary.to_csv(output / "summary.csv")
display(summary)
display(accepted_peptides[["sample", "sequence", "score"]].head())
print(f"Results saved to {output}")


The output directory contains `psms.csv`, `peptides.csv`, complete score
tables, the fitted QC checkpoints and FDR adjustment factors. Observed FDP is
reported only where accepted targets have reference labels. Protein-level
analysis and quantification are not executed by this notebook.
